# 2 — Adding Attention: Transformers for EEG

**Cutting-EEG Workshop 2026 · UCSD · Deep Learning EEG Methods and Practice**

Notebook 1 used a CNN. CNNs are built on an assumption: what matters is *local* — a
filter sees 25 samples at a time, and long-range structure only emerges after stacking
layers. Attention drops that assumption. Every timepoint can look at every other
timepoint directly.

This notebook is the bridge from CNNs to the foundation models in Notebook 3, because
**every EEG foundation model is a Transformer**. To fine-tune one, you need to know what
a token is for EEG.

### What we do here

1. Turn continuous EEG into **tokens** — the step that makes Transformers applicable
2. Build a compact Transformer encoder from scratch, so nothing is hidden
3. Train it, and compare honestly against the Notebook 1 CNN baseline
4. Visualize attention: which timepoints did the model actually use?
5. Use `EEGConformer` — Braindecode's production CNN+Transformer hybrid

### The honest framing

> On 280 training trials, a pure Transformer will probably **lose** to EEGNet. That is
> the expected result, not a bug. Attention has weak inductive bias, so it needs either
> lots of data or lots of pretraining. This notebook shows you *why* that is, which is
> exactly the argument for foundation models.

---
**Runtime: `Runtime → Change runtime type → T4 GPU`.**

In [ ]:
%pip install -q "braindecode[moabb]"

import IPython
print("Install finished — restarting the runtime.")
print("This 'crash' notice is expected. Continue at Section 1 below.")
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import mne
import matplotlib.pyplot as plt

mne.set_log_level("ERROR")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 20260916
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"torch {torch.__version__} · device {DEVICE}")

## 1 · Data

Same data and same pipeline as Notebook 1, condensed into one cell. If anything here is
unfamiliar, go back — this is Notebook 1 Sections 1–4 with no changes.

In [ ]:
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import (
    Preprocessor, preprocess, exponential_moving_standardize,
    create_windows_from_events,
)
from torch.utils.data import DataLoader

SUBJECT_ID = 3

dataset = MOABBDataset(dataset_name="BNCI2014_001", subject_ids=[SUBJECT_ID])
preprocess(dataset, [
    Preprocessor("pick_types", eeg=True, meg=False, eog=False),
    Preprocessor(lambda d: d * 1e6),
    Preprocessor("filter", l_freq=4.0, h_freq=38.0),
    Preprocessor(exponential_moving_standardize, factor_new=1e-3, init_block_size=1000),
], n_jobs=1)

sfreq = dataset.datasets[0].raw.info["sfreq"]
n_channels = len(dataset.datasets[0].raw.ch_names)

windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=int(-0.5 * sfreq),
    trial_stop_offset_samples=0,
    preload=True,
)

splits = windows_dataset.split("session")
train_set, test_set = splits["0train"], splits["1test"]

X0, y0, _ = train_set[0]
n_times = X0.shape[1]
class_names = windows_dataset.datasets[0].windows.event_id
n_classes = len(class_names)

BATCH_SIZE = 64
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print(f"{n_channels} channels × {n_times} samples ({n_times/sfreq:.1f}s), {n_classes} classes")
print(f"train {len(train_set)} · test {len(test_set)} windows")

## 2 · Tokenization: the core idea

A Transformer consumes a **sequence of vectors** (tokens). Text arrives pre-tokenized —
words are already discrete. EEG does not. So: what is a token for EEG?

**The naive answer — one token per timepoint — does not work.** Our window is 1125
samples. Attention is O(n²), so that's 1.27M attention weights per head per layer.
Worse, a single sample at 250 Hz carries almost no information; it's like tokenizing
text by individual letters.

**The working answer: patches.** Slice the signal into short segments (say 25 samples =
100 ms) and project each into an embedding. Every patch spans all channels, so a token
is *"what the whole scalp did during this 100 ms."*

```
EEG (22 ch × 1125 samples)
        │  slice into non-overlapping 25-sample patches
        ▼
45 patches, each (22 × 25)
        │  flatten and linearly project
        ▼
45 tokens, each a 64-dim vector      ← this is what the Transformer sees
```

100 ms is a deliberate choice. It's long enough to contain roughly one alpha cycle, and
short enough that the label doesn't change within a patch. This is the same trick ViT
plays on images, and the same one every EEG foundation model uses — the patch size is
one of the first hyperparameters you'll see in their configs.

In [ ]:
class PatchEmbedding(nn.Module):
    """Slice EEG into temporal patches and project each to `embed_dim`.

    A Conv1d with kernel_size == stride == patch_size is exactly
    "slice into non-overlapping patches, then apply a shared linear layer".
    """

    def __init__(self, n_channels, patch_size, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv1d(n_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):                 # x: (batch, channels, time)
        x = self.proj(x)                  #    (batch, embed_dim, n_patches)
        return x.transpose(1, 2)          #    (batch, n_patches, embed_dim)


PATCH_SIZE = 25          # 25 samples @ 250 Hz = 100 ms
EMBED_DIM = 64

_patcher = PatchEmbedding(n_channels, PATCH_SIZE, EMBED_DIM)
_demo = torch.randn(4, n_channels, n_times)
_tokens = _patcher(_demo)

print(f"in : {tuple(_demo.shape)}   (batch, channels, time)")
print(f"out: {tuple(_tokens.shape)}   (batch, n_patches, embed_dim)")
print(f"\n{n_times} samples / {PATCH_SIZE} per patch = {_tokens.shape[1]} tokens")
print(f"Attention matrix: {_tokens.shape[1]}² = {_tokens.shape[1]**2:,} weights per head")
print(f"Per-timepoint tokens would be: {n_times}² = {n_times**2:,}  — {n_times**2 // _tokens.shape[1]**2}× more")

## 3 · Positional encoding

Attention is permutation-invariant: shuffle the tokens and the output is identical. For
EEG that is fatal — a beta burst before the cue means something different from the same
burst after it.

So we add a position signal to each token. We use **learned** positional embeddings
(a trainable vector per position), which is what most EEG foundation models do.

We also prepend a **CLS token** — a learned vector that isn't tied to any timepoint. It
attends to all patches and accumulates a summary, and we classify from it. Borrowed from
BERT and standard in ViT.

In [ ]:
class EEGTransformer(nn.Module):
    """A compact Transformer encoder for EEG classification.

    Deliberately written out rather than imported, so every piece is visible.
    """

    def __init__(self, n_channels, n_times, n_classes,
                 patch_size=25, embed_dim=64, depth=4, n_heads=4,
                 mlp_ratio=2.0, dropout=0.3):
        super().__init__()

        self.patch_embed = PatchEmbedding(n_channels, patch_size, embed_dim)
        n_patches = n_times // patch_size

        # CLS token + learned positions (+1 for the CLS slot)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.dropout = nn.Dropout(dropout)

        # norm_first=True (pre-LN) trains far more stably on small data
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, n_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                             # (B, P, D)
        cls = self.cls_token.expand(B, -1, -1)              # (B, 1, D)
        x = torch.cat([cls, x], dim=1)                      # (B, P+1, D)
        x = self.dropout(x + self.pos_embed)
        x = self.encoder(x)
        return self.head(self.norm(x[:, 0]))                # classify from CLS


model = EEGTransformer(n_channels, n_times, n_classes,
                       patch_size=PATCH_SIZE, embed_dim=EMBED_DIM,
                       depth=4, n_heads=4, dropout=0.3).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"EEGTransformer: {n_params:,} parameters")
print(f"(EEGNet was ~2,000 — this is {n_params // 2000}× larger on the same data.)")

with torch.no_grad():
    out = model(torch.randn(2, n_channels, n_times, device=DEVICE))
print(f"\nShape check: (2, {n_channels}, {n_times}) -> {tuple(out.shape)}")

### Why `norm_first=True` matters

Post-LN Transformers (the original 2017 design) need learning-rate warmup to train
stably. Pre-LN (`norm_first=True`) is much more forgiving. On EEG-sized datasets, this
one flag is often the difference between training and diverging — worth remembering when
someone's Transformer "just doesn't learn."

## 4 · Training

Same loop shape as Notebook 1, with two changes that matter for Transformers:

- **Lower learning rate** (1e-3 vs 6e-3). Attention is more sensitive.
- **Warmup.** 10 epochs of linear ramp before cosine decay. Early in training, attention
  weights are near-uniform and gradients are noisy; a big early step can wreck them.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import math, time

N_EPOCHS = 80
LR = 1e-3
WARMUP_EPOCHS = 10
WEIGHT_DECAY = 0.05          # heavier than the CNN — more parameters, same data

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, N_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = LambdaLR(optimizer, lr_lambda)


def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_correct, n_total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for X, y, _ in loader:
            X, y = X.to(device).float(), y.to(device).long()
            logits = model(X)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # Transformers spike
                optimizer.step()
            total_loss += loss.item() * len(y)
            n_correct += (logits.argmax(1) == y).sum().item()
            n_total += len(y)

    return total_loss / n_total, n_correct / n_total

In [ ]:
history = {"train_acc": [], "test_acc": [], "train_loss": [], "test_loss": []}
start = time.time()

print(f"Training {N_EPOCHS} epochs ({WARMUP_EPOCHS} warmup) on {DEVICE}\n")
for epoch in range(N_EPOCHS):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    te_loss, te_acc = run_epoch(model, test_loader, criterion, None)
    scheduler.step()

    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["test_loss"].append(te_loss);  history["test_acc"].append(te_acc)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"epoch {epoch+1:3d}/{N_EPOCHS} | "
              f"train {tr_loss:.3f}/{tr_acc:.1%} | test {te_loss:.3f}/{te_acc:.1%} | "
              f"lr {scheduler.get_last_lr()[0]:.1e}")

transformer_best = max(history["test_acc"])
print(f"\nDone in {time.time()-start:.0f}s")
print(f"Best test accuracy: {transformer_best:.1%}  (chance {1/n_classes:.0%})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ep = range(1, N_EPOCHS + 1)

ax1.plot(ep, history["train_loss"], label="train", lw=2)
ax1.plot(ep, history["test_loss"], label="test", lw=2)
ax1.axvline(WARMUP_EPOCHS, ls=":", c="gray", label="warmup ends")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.set_title("Loss")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, history["train_acc"], label="train", lw=2)
ax2.plot(ep, history["test_acc"], label="test", lw=2)
ax2.axhline(1/n_classes, ls="--", c="gray", label="chance")
ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy"); ax2.set_title("Accuracy")
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("EEG Transformer — watch the train/test gap", y=1.02)
plt.tight_layout(); plt.show()

gap = history["train_acc"][-1] - history["test_acc"][-1]
print(f"Train − test gap: {gap:.1%}")
if gap > 0.25:
    print("Large gap: the Transformer memorized the training set.")
    print("With ~280 trials and no pretraining, this is the expected outcome.")

## 5 · What did attention look at?

The interpretability payoff. We extract the CLS token's attention over patches — literally
"which 100 ms segments did the model consult to make its decision?"

For motor imagery, we'd hope to see weight *after* the cue, during the imagery period.

**A caution worth stating**, because it is widely ignored in the literature: attention
weights are **not** a faithful explanation. High attention doesn't prove causal
importance — a token can be attended to and still not affect the output. Treat this as a
hypothesis generator, not evidence. If you want to claim a timepoint matters, ablate it
and measure.

In [ ]:
def get_cls_attention(model, x):
    """Return CLS-token attention over patches from the final encoder layer.

    Why this isn't a one-liner: `nn.TransformerEncoderLayer` throws attention
    weights away (`_sa_block` calls attention with `need_weights=False`), and
    `nn.TransformerEncoder` may take a fused fast path that skips the layer's
    Python body entirely — so a monkey-patched hook can silently never fire.

    The reliable approach is to run the encoder stack ourselves: feed the input
    through every layer but the last, then call the last layer's attention
    directly with `need_weights=True`. No patching, no fast-path surprises.
    """
    model.eval()

    with torch.no_grad():
        # Rebuild the token sequence exactly as EEGTransformer.forward does.
        B = x.shape[0]
        h = model.patch_embed(x)
        cls = model.cls_token.expand(B, -1, -1)
        h = torch.cat([cls, h], dim=1) + model.pos_embed

        # Run all layers except the last.
        for layer in model.encoder.layers[:-1]:
            h = layer(h)

        # Last layer: apply its pre-attention norm (norm_first=True), then
        # call attention explicitly so we get the weight matrix back.
        last = model.encoder.layers[-1]
        h_norm = last.norm1(h) if last.norm_first else h
        _, attn_weights = last.self_attn(
            h_norm, h_norm, h_norm,
            need_weights=True, average_attn_weights=True,
        )

    # (batch, tokens, tokens) -> CLS row, patches only (drop the CLS-to-CLS entry)
    return attn_weights[:, 0, 1:].cpu().numpy()


X_batch, y_batch, _ = next(iter(test_loader))
attn = get_cls_attention(model, X_batch.to(DEVICE).float())
print(f"Attention: {attn.shape}  (batch, n_patches)")

In [ ]:
patch_sec = PATCH_SIZE / sfreq
times = np.arange(attn.shape[1]) * patch_sec - 0.5   # window starts 0.5s pre-cue

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)

mean_attn = attn.mean(0)
ax1.plot(times, mean_attn, lw=2, color="darkblue")
ax1.fill_between(times, mean_attn, alpha=0.3, color="darkblue")
ax1.axvline(0, c="red", ls="--", lw=2, label="cue onset")
ax1.axhline(1/attn.shape[1], c="gray", ls=":", label="uniform attention")
ax1.set_ylabel("mean CLS attention"); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title("Where the model looked (averaged over the batch)")

im = ax2.imshow(attn, aspect="auto", cmap="viridis",
                extent=[times[0], times[-1], attn.shape[0], 0])
ax2.axvline(0, c="red", ls="--", lw=2)
ax2.set_xlabel("time relative to cue (s)"); ax2.set_ylabel("trial")
ax2.set_title("Per-trial attention")
plt.colorbar(im, ax=ax2, label="weight")

plt.tight_layout(); plt.show()

pre, post = mean_attn[times < 0].mean(), mean_attn[times >= 0].mean()
print(f"Mean attention pre-cue : {pre:.4f}")
print(f"Mean attention post-cue: {post:.4f}")
print(f"Ratio post/pre: {post/pre:.2f}×")
print("\n>1 means the model concentrated on the imagery period — the sensible outcome.")
print("≈1 means attention stayed near-uniform, which is common when data is scarce.")

## 6 · EEGConformer: the pragmatic answer

Our from-scratch Transformer was for understanding. In practice you'd use a **hybrid**:
convolutions first to extract local features, attention on top for long-range structure.

This gets you both inductive biases — the CNN front end handles "EEG is oscillatory and
spatially structured," and attention handles "what happened 2 seconds ago matters." On
small datasets, hybrids reliably beat pure Transformers, and `EEGConformer` is
Braindecode's well-tested implementation.

> **API note:** `EEGConformer` takes `n_outputs` **first**, not `n_chans`. Always pass
> these by keyword — positional args will silently misassign.

In [ ]:
from braindecode.models import EEGConformer

conformer = EEGConformer(
    n_outputs=n_classes,
    n_chans=n_channels,
    n_times=n_times,
    final_fc_length="auto",
).to(DEVICE)

print(f"EEGConformer: {sum(p.numel() for p in conformer.parameters()):,} parameters")

opt_c = AdamW(conformer.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
sched_c = LambdaLR(opt_c, lr_lambda)
crit_c = nn.CrossEntropyLoss(label_smoothing=0.1)

conformer_acc = []
print()
for epoch in range(N_EPOCHS):
    run_epoch(conformer, train_loader, crit_c, opt_c)
    _, te_acc = run_epoch(conformer, test_loader, crit_c, None)
    sched_c.step()
    conformer_acc.append(te_acc)
    if (epoch + 1) % 20 == 0:
        print(f"epoch {epoch+1:3d} | test {te_acc:.1%}")

print(f"\nEEGConformer best: {max(conformer_acc):.1%}")

## 7 · The comparison

Let's also train EEGNet here, so all three numbers come from the same split, the same
seed, and the same machine. Comparing against a number you remember from another
notebook is how bad benchmarks happen.

In [ ]:
from braindecode.models import EEGNet

eegnet = EEGNet(n_chans=n_channels, n_outputs=n_classes,
                n_times=n_times, final_conv_length="auto").to(DEVICE)
opt_e = AdamW(eegnet.parameters(), lr=6e-3, weight_decay=1e-4)
sched_e = torch.optim.lr_scheduler.CosineAnnealingLR(opt_e, T_max=N_EPOCHS)
crit_e = nn.CrossEntropyLoss()

eegnet_acc = []
for epoch in range(N_EPOCHS):
    run_epoch(eegnet, train_loader, crit_e, opt_e)
    _, te = run_epoch(eegnet, test_loader, crit_e, None)
    sched_e.step()
    eegnet_acc.append(te)

results = {
    "EEGNet (CNN)": (max(eegnet_acc), sum(p.numel() for p in eegnet.parameters())),
    "EEGTransformer (ours)": (transformer_best, n_params),
    "EEGConformer (hybrid)": (max(conformer_acc), sum(p.numel() for p in conformer.parameters())),
}

print(f"{'Model':<26}{'best acc':>10}{'params':>12}")
print("-" * 48)
for name, (acc, npar) in results.items():
    print(f"{name:<26}{acc:>9.1%}{npar:>12,}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

for name, curve in [("EEGNet (CNN)", eegnet_acc),
                    ("EEGTransformer", history["test_acc"]),
                    ("EEGConformer", conformer_acc)]:
    ax1.plot(range(1, len(curve) + 1), curve, lw=2, label=name)
ax1.axhline(1/n_classes, ls="--", c="gray", label="chance")
ax1.set_xlabel("epoch"); ax1.set_ylabel("test accuracy")
ax1.set_title("Test accuracy over training"); ax1.legend(); ax1.grid(alpha=0.3)

names = list(results.keys())
accs = [results[n][0] for n in names]
pars = [results[n][1] for n in names]
ax2.scatter(pars, accs, s=180, c=["tab:blue", "tab:orange", "tab:green"], zorder=3)
for n, p, a in zip(names, pars, accs):
    ax2.annotate(n.split(" (")[0], (p, a), textcoords="offset points",
                 xytext=(0, 12), ha="center", fontsize=9)
ax2.set_xscale("log"); ax2.axhline(1/n_classes, ls="--", c="gray")
ax2.set_xlabel("parameters (log)"); ax2.set_ylabel("best test accuracy")
ax2.set_title("More parameters ≠ better, when data is fixed"); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Recap

**If the Transformer lost to EEGNet, that is the correct result** — and it is the whole
argument for the next notebook.

A CNN has strong inductive bias: it *assumes* locality and translation equivariance, both
true of EEG. Those assumptions are free knowledge, so it can learn from 280 trials. A
Transformer assumes almost nothing and must learn structure from data — powerful when
data is plentiful, a liability when it isn't.

You have three ways out:

1. **More data** — pool subjects, pool datasets
2. **Put the bias back** — hybrids like EEGConformer
3. **Pretrain elsewhere** — learn structure on thousands of hours of unlabeled EEG, then
   fine-tune on your 280 trials

Option 3 is what an **EEG foundation model** is. That's Notebook 3.

### Try it yourself

1. Change `PATCH_SIZE` to 10 and 50. Trade-off: more tokens = finer resolution but
   quadratically more compute and more overfitting.
2. Set `depth=1`. Does a shallower Transformer generalize better here?
3. Load 4 subjects (`subject_ids=[1,2,3,4]`). Does the Transformer close the gap as data
   grows? This is the scaling argument in miniature.